## Импорты

In [1]:
CONFIG_NAME = "custom_correction.yaml"
#CONFIG_NAME = "custom_CNN_RNN.yaml"

In [2]:
import sys
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path

# Допустим, что ноутбук находится в той же директории, что и папка acoustic/
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation_utils").setLevel(logging.ERROR)

In [3]:
from acoustic.utils.config import load_config
from acoustic.dataset.load_dataset import load_and_prepare_dataset
from acoustic.models.load_model import build_model
from acoustic.training.load_metrics import load_metrics
from acoustic.training.callbacks import get_callback
from acoustic.training import get_trainer_class


## Загрузка конфигурации

In [4]:
CONFIG_PATH = f"acoustic/configs/{CONFIG_NAME}"

cfg = load_config(CONFIG_PATH, overrides=None)

print("Configuration loaded")

Configuration loaded


## Загрузка датасета

In [5]:
dataset = load_and_prepare_dataset(cfg)

Loading dataset from disk:   0%|          | 0/18 [00:00<?, ?it/s]

Acoustic inference: 100%|██████████| 7813/7813 [16:23<00:00,  7.95it/s]


Saving the dataset (0/1 shards):   0%|          | 0/474905 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Applying filters: 0it [00:00, ?it/s]


## Инициализация модели

In [6]:
model, processor, data_collator = build_model(cfg)

print("Model built")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
You are using a model of type mt5 to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


Model built


## Создание и загрузка метрик, callbacks, trainer

In [7]:
metrics_list = load_metrics(cfg['training']['metrics'])
print(f"Metrics: {cfg['training']['metrics']}")

callbacks = []
for cb_name in cfg['training']['callbacks']:
    callbacks.append(get_callback(cb_name))

Metrics: ['wer', 'cer', 'f1', 'detailed_stats', 'ser', 'space_wer']


In [8]:

trainer_name = cfg['training'].get('trainer', 'BaseTrainer')
TrainerClass = get_trainer_class(trainer_name)

trainer = TrainerClass(
    cfg=cfg,
    model=model,
    processor=processor,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    metrics=metrics_list,
    callbacks=callbacks,
    data_collator=data_collator
)

## Обучение

In [ ]:
print("Starting training")
trainer.train()

Starting training


Training:   6%|▋         | 4704/74205 [07:49<1:58:59,  9.73step/s]

## Проверка

In [ ]:
print("Demo on validation examples")
import random
import torch
from acoustic.models import get_generate_method

eval_dataset = dataset['validation']

if eval_dataset and len(eval_dataset) > 0:
    indices = random.sample(range(len(eval_dataset)), min(10, len(eval_dataset)))
    device = next(model.parameters()).device
    model.eval()
    builder_key = cfg['model']['builder']
    generate_fn = get_generate_method(builder_key)

    for i in indices:
        example = eval_dataset[i]

        # Ветка для текстового корректора
        if builder_key == "correction_model":
            stt_text = "исправь: " + example["stt_text"]
            ref_text = example["reference"]

            inputs = processor(stt_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
            input_data = inputs["input_ids"].to(device)

            with torch.no_grad():
                predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" ASR output : {stt_text}")
            print(f" Corrected  : {pred_text}")
            print(f" Reference  : {ref_text}")

        # Ветка для всех акустических моделей
        else:
            audio_array = example["audio"]["array"]
            ref_text = example["sentence"]

            inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
            input_key = "input_features" if "input_features" in inputs else "input_values"
            input_data = inputs[input_key].to(device)

            with torch.no_grad():
                predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" Reference: {ref_text}")
            print(f" Prediction: {pred_text}")
else:
    print("No validation dataset for demo.")

Demo on validation examples

Example 1629:
 Reference: закажи чипсы
 Prediction: закажичипцы

Example 4465:
 Reference: трансляция апл сити против ливер
 Prediction: трансляция апл сити против лигер

Example 3437:
 Reference: у тебя в списке есть киношка с актрисой дженнифер лоуренс
 Prediction: у тебя в списке есть киношка с актрисйджни фр лоарен

Example 1806:
 Reference: афина поставь звук потише
 Prediction: афина поставь зупатиша

Example 3680:
 Reference: раиф сипачев
 Prediction: раи все почев

Example 4828:
 Reference: загородная жизнь на смотрешке
 Prediction: загрудная жизнь на смотрешке 

Example 2279:
 Reference: на ютьюбе токинг том энд френдс
 Prediction: най ютьюбе токинтон энтфрэнс

Example 54:
 Reference: анастасия витальевна пантелеева
 Prediction: анстасиеюталена анчеа

Example 1308:
 Reference: найди лучшие фильмы жанра спорт
 Prediction: найди лучше фильм ы жан ра спорт

Example 3463:
 Reference: продолжить
 Prediction: продолжить
